# Tool catalog: processes and terminals

Run two bounded subprocesses, then create one disposable tmux pane to inspect. Setup uses a private tmux socket directory, so the examples do not list or read your normal terminal sessions. Only this notebook-created pane is read. Cleanup removes it.


In [1]:
from nbinlineai.tools import run_python, run_shell, tmux_sessions, tmux_read
from tempfile import TemporaryDirectory
import os, subprocess, uuid
catalog_tmux_temp = TemporaryDirectory(prefix="nbinlineai-tmux-")
catalog_prior_tmux_dir = os.environ.get("TMUX_TMPDIR")
catalog_prior_tmux = os.environ.pop("TMUX", None)
os.environ["TMUX_TMPDIR"] = catalog_tmux_temp.name
def catalog_restore_tmux_environment():
    if catalog_prior_tmux is None:
        os.environ.pop("TMUX", None)
    else:
        os.environ["TMUX"] = catalog_prior_tmux
    if catalog_prior_tmux_dir is None:
        os.environ.pop("TMUX_TMPDIR", None)
    else:
        os.environ["TMUX_TMPDIR"] = catalog_prior_tmux_dir
catalog_tmux_name = "nbinlineai_catalog_" + uuid.uuid4().hex[:10]
try:
    subprocess.run(["tmux", "new-session", "-d", "-s", catalog_tmux_name], check=True)
    catalog_pane_id = subprocess.check_output(["tmux", "list-panes", "-t", catalog_tmux_name, "-F", "#{pane_id}"], text=True).strip()
    subprocess.run(["tmux", "send-keys", "-t", catalog_pane_id, "printf catalog-pane-ready", "Enter"], check=True)
except Exception:
    try:
        subprocess.run(["tmux", "kill-session", "-t", catalog_tmux_name], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except OSError:
        pass
    catalog_restore_tmux_environment()
    catalog_tmux_temp.cleanup()
    raise
print("Owned disposable pane:", catalog_pane_id)


Owned disposable pane: %0


### Offered tools

- &`run_python`
- &`run_shell`
- &`tmux_sessions`
- &`tmux_read`


## run_python


In [2]:
print(run_python("print(2 + 3)", timeout=5))


Exit code: 0.
Working directory: [temporary local path]

5



## run_shell


In [3]:
print(run_shell("echo catalog-ready", timeout=5))


Exit code: 0.
Working directory: [temporary local path]

catalog-ready



## Ask AI to run and inspect the same small tasks

The Python and shell tools run subprocesses; the tmux tools inspect only the owned pane from setup.


Use &`run_python` for `print(2 + 3)` and &`run_shell` for `echo catalog-ready`, both with a five-second timeout. State each actual exit status and output.


- `run_python`: exit status 0; output `5`.
- `run_shell`: exit status 0; output `catalog-ready`.

**Observed live tool calls for the answer above** (actual subscription events; private fields shortened).

| Tool | Result state | Submitted arguments | Observed result |
| --- | --- | --- | --- |
| `run_python` | completed | {"code":"print(2 + 3)","timeout":5} | 'Exit code: 0.\nWorking directory: [local path]' |
| `run_shell` | completed | {"command":"echo catalog-ready","timeout":5} | 'Exit code: 0.\nWorking directory: [local path]' |


## tmux_sessions


In [4]:
print(tmux_sessions())


Exit code: 0.
Working directory: [temporary local path]

nbinlineai_catalog_969e39bbfd:0.0 %0 zsh



Use &`tmux_sessions` to find the owned pane $`catalog_pane_id`. State its session and pane ID; ignore other panes.


Session: `nbinlineai_catalog_969e39bbfd`; pane ID: `%0`.

**Observed live tool calls for the answer above** (actual subscription events; private fields shortened).

| Tool | Result state | Submitted arguments | Observed result |
| --- | --- | --- | --- |
| `tmux_sessions` | completed | {} | 'Exit code: 0.\nWorking directory: [local path] %0 zsh\n' |


## tmux_read


In [5]:
print(tmux_read(catalog_pane_id, lines=10))


Exit code: 0.
Working directory: [temporary local path]

printf catalog-pane-ready

rahul in T/nbinlineai-worked-mh7jflqg/root via 🐍 v3.12.10
❯ printf catalog-pane-ready
catalog-pane-ready%

rahul in T/nbinlineai-worked-mh7jflqg/root via 🐍 v3.12.10
❯



















Use &`tmux_read` with pane $`catalog_pane_id` and `lines=10`. Report whether its screen contains `catalog-pane-ready`. Do not send input.


Yes, pane `%0` contains `catalog-pane-ready`.

**Observed live tool calls for the answer above** (actual subscription events; private fields shortened).

| Tool | Result state | Submitted arguments | Observed result |
| --- | --- | --- | --- |
| `tmux_read` | completed | {"pane":"%0","lines":10} | 'Exit code: 0.\nWorking directory: [local path] catalog-pane-ready\n\nrahul in T/nbinlineai-worked-mh7jflqg/root via 🐍 v3.12.10\n❯ printf catalog-pane-ready\ncatalog-pane-ready%\n\nrahul in T/nbinlineai-worked-mh7jflqg/root via 🐍 v3.12.10\n❯\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n' |


In [6]:
try:
    subprocess.run(["tmux", "kill-session", "-t", catalog_tmux_name], check=True)
finally:
    catalog_restore_tmux_environment()
    catalog_tmux_temp.cleanup()
print("Owned tmux session removed.")


Owned tmux session removed.
